In [1]:
# Install required libraries
import sys
!{sys.executable} -m pip install -q openai langchain langchain-community chromadb pypdf
!{sys.executable} -m pip install -q -U langchain-openai langchain-text-splitters langchain-classic streamlit


In [2]:
import os
# Set your API key before running embedding / LLM cells
# os.environ["OPENAI_API_KEY"] = "your_openai_api_key"
print("Environment ready")


Environment ready


# Step-1: Document Loading


In [3]:
import pandas as pd
from langchain_core.documents import Document

df = pd.read_csv("ecommerce_products_dataset.csv")
df.head()


,product_id,product_name,category,brand,price,rating,stock,discount_percent,seller,description
0,P10001,Mamaearth Moisturizer,Beauty,Mamaearth,1093.14,4.5,241,10,DailyDeals,Mamaearth Moisturizer is a budget-friendly bea...
1,P10002,Maybelline Makeup Brush Set,Beauty,Maybelline,3294.69,3.2,22,5,RetailHub,Maybelline Makeup Brush Set is a lightweight b...
2,P10003,Puma Men Cotton T-Shirt,Fashion,Puma,5423.15,3.3,162,30,QuickCart,Puma Men Cotton T-Shirt is a premium fashion p...
3,P10004,Penguin Productivity Habits,Books,Penguin,683.68,4.3,204,10,ValueMart,Penguin Productivity Habits is a stylish books...
4,P10005,McGraw Hill Travel Stories,Books,McGraw Hill,1307.56,3.2,163,5,DailyDeals,McGraw Hill Travel Stories is a high-performan...


In [4]:
documents = []
for _, row in df.iterrows():
    content = f'''
Product ID: {row["product_id"]}
Product Name: {row["product_name"]}
Category: {row["category"]}
Brand: {row["brand"]}
Price: {row["price"]}
Rating: {row["rating"]}
Stock: {row["stock"]}
Discount Percent: {row["discount_percent"]}
Seller: {row["seller"]}
Description: {row["description"]}
'''
    documents.append(
        Document(
            page_content=content,
            metadata={
                "product_id": row["product_id"],
                "product_name": row["product_name"],
                "category": row["category"],
                "brand": row["brand"]
            }
        )
    )

print("Total product documents:", len(documents))
print(documents[0])


Total product documents: 1000
page_content='
Product ID: P10001
Product Name: Mamaearth Moisturizer
Category: Beauty
Brand: Mamaearth
Price: 1093.14
Rating: 4.5
Stock: 241
Discount Percent: 10
Seller: DailyDeals
Description: Mamaearth Moisturizer is a budget-friendly beauty product with ergonomic design. Ideal for customers looking for reliable quality and value.
' metadata={'product_id': 'P10001', 'product_name': 'Mamaearth Moisturizer', 'category': 'Beauty', 'brand': 'Mamaearth'}


In [5]:
full_text = ""
for doc in documents:
    full_text += doc.page_content + "\n"

print("Documents:", len(documents))
print("Lines:", len(full_text.split("\n")))
print("Words:", len(full_text.split()))
print("Characters:", len(full_text))


Documents: 1000
Lines: 12001
Words: 44447
Characters: 316250


# Step-2: Split the data into Chunks


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = text_splitter.split_documents(documents)
print("Total chunks:", len(chunks))


c:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks: 1850


In [7]:
print(chunks[1])


page_content='Seller: DailyDeals
Description: Mamaearth Moisturizer is a budget-friendly beauty product with ergonomic design. Ideal for customers looking for reliable quality and value.' metadata={'product_id': 'P10001', 'product_name': 'Mamaearth Moisturizer', 'category': 'Beauty', 'brand': 'Mamaearth'}


# Step-3: Creating embeddings


In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()
embeddings


In [ ]:
sample_embedding = embeddings.embed_query("Show premium electronics with high ratings")
print("Embedding length:", len(sample_embedding))


In [ ]:
sample_docs = [
    "Wireless headphones with high ratings and premium build quality.",
    "Budget-friendly home products with good customer reviews."
]

embedded_vectors = embeddings.embed_documents(sample_docs)
print("Number of vectors:", len(embedded_vectors))
print("Vector dimension:", len(embedded_vectors[0]))


# Step-4: Storing in Vector Stores


In [ ]:
from langchain_community.vectorstores import Chroma

ecommerce_db = Chroma.from_documents(
    chunks,
    embeddings,
    persist_directory="ecommerce_rag_db"
)
ecommerce_db.persist()
print("Vector DB created successfully")


# Step-5: Retrieval


In [ ]:
retriever = ecommerce_db.as_retriever()
result = retriever.invoke("Show electronics products with premium quality and good ratings")
result


In [ ]:
for i in range(len(result)):
    print(result[i].metadata)


In [ ]:
retriever = ecommerce_db.as_retriever()
result = retriever.invoke("Find beauty products with premium feel and strong ratings")
result


# MultiQueryRetriever


In [ ]:
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_openai import OpenAI

llm = OpenAI(temperature=0)


In [ ]:
llm_based_retriever = MultiQueryRetriever.from_llm(
    retriever=ecommerce_db.as_retriever(),
    llm=llm
)


In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)


In [ ]:
question1 = "Suggest budget-friendly electronics with good ratings"
rel_docs1 = llm_based_retriever.invoke(question1)
rel_docs1


In [ ]:
question2 = "Find travel-friendly products for frequent shoppers"
rel_docs2 = llm_based_retriever.invoke(question2)
rel_docs2


# Contextual Compression


In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

llm = OpenAI(temperature=0)
compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ecommerce_db.as_retriever()
)


In [ ]:
compressed_docs = compression_retriever.invoke(question1)
compressed_docs[0].metadata


# RetrievalQA Chain


In [ ]:
from langchain_classic.chains import RetrievalQA

llm = OpenAI(temperature=0)

Q_AChain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=llm_based_retriever
)


In [ ]:
query = "Recommend a few highly rated fashion products for daily use and summarize why they fit."
docs = Q_AChain({"query": query})
docs["result"]


In [ ]:
print(Q_AChain.combine_documents_chain.llm_chain.prompt.template)


Validation of the Results


In [ ]:
test_data = pd.read_csv("Ecommerce_QA_Validation.csv")
test_data.head()


In [ ]:
test_qa_pairs = []
for index, row in test_data.iterrows():
    question = row["Question"]
    answer = row["Expected_Answer"]
    test_qa_pairs.append({"question": question, "answer": answer})

print(test_qa_pairs)


## Simple fallback retrieval without embeddings


In [ ]:
def build_product_text(dataframe):
    text_cols = [
        "product_id", "product_name", "category", "brand", "price",
        "rating", "stock", "discount_percent", "seller", "description"
    ]
    for col in text_cols:
        dataframe[col] = dataframe[col].fillna("").astype(str)

    dataframe["product_text"] = (
        "Product ID: " + dataframe["product_id"] + ". " +
        "Product Name: " + dataframe["product_name"] + ". " +
        "Category: " + dataframe["category"] + ". " +
        "Brand: " + dataframe["brand"] + ". " +
        "Price: " + dataframe["price"] + ". " +
        "Rating: " + dataframe["rating"] + ". " +
        "Stock: " + dataframe["stock"] + ". " +
        "Discount Percent: " + dataframe["discount_percent"] + ". " +
        "Seller: " + dataframe["seller"] + ". " +
        "Description: " + dataframe["description"]
    )
    return dataframe

df = build_product_text(df)
df[["product_name", "product_text"]].head(2)


In [ ]:
def simple_retrieve(dataframe, query, top_k=5):
    query_terms = [term.strip().lower() for term in query.replace("/", " ").replace(",", " ").split() if term.strip()]
    matches = []

    for _, row in dataframe.iterrows():
        text = str(row.get("product_text", "")).lower()
        score = sum(1 for term in query_terms if term in text)
        if score > 0:
            try:
                rating = float(row.get("rating", 0))
            except:
                rating = 0
            matches.append((score, row.get("product_name", ""), row.get("category", ""), rating, row.get("price", ""), row.get("description", "")))

    matches.sort(key=lambda x: (x[0], x[3]), reverse=True)
    return matches[:top_k]


In [ ]:
query = "premium electronics high rating travel friendly"
results = simple_retrieve(df, query, top_k=5)

for item in results:
    print(item)
